# Cuaderno 03 — Análisis espacial del recurso solar por celdas de la grilla NASA POWER

**Proyecto de Grado — Maestría en Ciencia de Datos**
**Pontificia Universidad Javeriana Cali — Grupo 8**
**Autores:** Laura García Salamanca, Daniel Sebastián Rosero Usamá, Juan Villalobos Mora

---

## Para qué existe este cuaderno

Responde a tres observaciones de la dirección del trabajo de grado:

1. **Verificar la resolución real de la fuente.** El documento afirma que las variables de
   irradiancia tienen resolución de 1° × 1°. Aquí se comprueba empíricamente, descargando
   un vecindario de celdas y observando a qué distancia cambia el valor.
2. **Construir el mapa de grillas.** Un mapa del entorno de Pasto, dibujado a la resolución
   con la que efectivamente se trabaja, coloreado por estadísticas de la serie de irradiancia
   de cada celda: promedio, percentil 25, percentil 75 y variabilidad.
3. **Comparar celda contra estación.** Para varias celdas del vecindario, la serie temporal
   diaria con su tendencia, superpuesta a la serie de la estación CESMAG. Esto responde a la
   pregunta de en qué celda cae la estación y cuánto se parece esa celda a lo que el
   piranómetro realmente mide.

## Qué produce

| Archivo | Contenido |
|---|---|
| `nasa_power_grillas_diario.csv` | Serie diaria de irradiancia por celda, período completo |
| `tabla_08_estadisticas_por_celda.csv` | Estadísticas descriptivas de cada celda |
| `tabla_08_comparacion_celda_estacion.csv` | Correlación, sesgo y error de cada celda frente a la estación |
| `fig_08a_mapa_estadisticas.pdf` | Mapa de cuatro paneles con las estadísticas por celda |
| `fig_08b_series_por_celda.pdf` | Series diarias por celda frente a la estación |
| `fig_08c_verificacion_resolucion.pdf` | Evidencia de la resolución nativa del producto |

## Antes de ejecutar

Requiere que el cuaderno 02 haya generado `dataset_final_interpolacion.csv`, del que se toma
la serie de la estación para la comparación. La descarga completa tarda entre 5 y 15 minutos.


## 0. Importaciones y configuración de estilo

In [ ]:
!pip install requests pandas numpy matplotlib geopandas contextily --quiet

import os
import time
import json
import warnings
from datetime import datetime

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

# --- Estilo unificado del documento -----------------------------------------
# Criterios que se aplican en todas las figuras de este cuaderno:
#   - Sin titulo interno: el titulo va en el caption de LaTeX.
#   - Etiquetas de ejes y leyendas en espaniol y en lenguaje natural.
#   - Tamanio de letra suficiente para leerse al 300 % de zoom.
#   - Salida vectorial en PDF, para que no pixele al ampliar.
# El estilo de las figuras (paleta y tamanos de letra) se define en la celda
# siguiente, la celda de estilo, para que sea identico en los 11 cuadernos.


print('Librerias importadas:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


### Estilo unico de figuras

Esta celda fija la **paleta de colores** y los **tamanos de letra** de todas las figuras del cuaderno. Es la misma celda en los once cuadernos, de modo que cualquier figura del trabajo de grado sale con el mismo aspecto.


In [ ]:
# @@ESTILO_TG@@  (marca unica: no editar esta linea)
# =============================================================================
#  ESTILO UNICO DE FIGURAS DEL TRABAJO DE GRADO
#  Esta celda define la paleta y la tipografia de TODAS las figuras.
#  Es identica en los 11 cuadernos: cambiarla aqui cambia solo este cuaderno,
#  por eso si se modifica hay que modificarla en todos.
#
#  Paleta categorica anclada en el azul. Validada con el verificador de
#  accesibilidad de color (separacion OKLab bajo protanopia y deuteranopia,
#  piso de croma, banda de luminosidad y contraste contra fondo blanco):
#  todos los pares de colores pasan, no solo los contiguos.
# =============================================================================
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from cycler import cycler
try:
    import seaborn as sns
except Exception:
    sns = None

# --- 1. Paleta categorica (orden FIJO: la ranura 1 es siempre la serie principal)
C_AZUL    = '#0C60A3'   # ranura 1 - serie principal / observado / T1 / V1
C_AMBAR   = '#D4771A'   # ranura 2 - segunda serie / modelo / T2 / V2
C_CELESTE = '#269AE6'   # ranura 3 - tercera serie / T3 / V3
C_TEJA    = '#963E35'   # ranura 4 - cuarta serie
C_VERDEAZ = '#0D8F73'   # ranura 5 - quinta serie
PALETA_TG = [C_AZUL, C_AMBAR, C_CELESTE, C_TEJA, C_VERDEAZ]

# --- 2. Tonos de apoyo de la rampa azul (rellenos, bandas, sombreados)
C_AZUL_CLARO  = '#81B4EA'
C_AZUL_MEDIO  = '#5A9BDC'
C_AZUL_OSCURO = '#0A3765'

# --- 3. Tinta neutra: texto, ejes, lineas de referencia. NUNCA es una serie.
C_TEXTO      = '#1A1A1A'
C_GRIS       = '#6E6E6E'
C_GRIS_CLARO = '#C9C9C9'

# --- 4. Mapas de color continuos
#   'tg_azul' : secuencial de un solo tono, claro -> oscuro (magnitudes)
#   'tg_div'  : divergente azul <-> teja con gris neutro en el centro
#               (mantiene la lectura de 'RdBu_r': negativo azul, positivo calido)
_SECUENCIAL = ['#F4F9FF', '#D8E8FA', '#B4D2F4', '#81B4EA', '#5A9BDC',
               '#3581C9', '#1667AC', '#0B4E89', '#0A3765']
_DIVERGENTE = ['#0A3765', '#1667AC', '#5A9BDC', '#AFCEEC', '#F2F1EF',
               '#EFC9AC', '#DC9A5A', '#B4642C', '#7E3A2C']
CMAP_SEC = LinearSegmentedColormap.from_list('tg_azul', _SECUENCIAL)
CMAP_DIV = LinearSegmentedColormap.from_list('tg_div',  _DIVERGENTE)

def _registrar_cmap(cm_obj):
    """Registra el mapa por nombre para poder usar cmap='tg_azul'."""
    try:
        mpl.colormaps.register(cm_obj, force=True)          # matplotlib >= 3.6
    except Exception:
        try:
            mpl.cm.register_cmap(cmap=cm_obj)               # matplotlib antiguo
        except Exception:
            pass

for _c in (CMAP_SEC, CMAP_DIV):
    _registrar_cmap(_c)

def escala_azul(n, ini=0.375, fin=1.0):
    """n colores tomados de la rampa azul. Para variables ORDENADAS (anios,
    meses, horizontes): el orden se lee en el tono, de claro a oscuro."""
    if n <= 1:
        return [C_AZUL]
    paso = (fin - ini) / (n - 1)
    return [CMAP_SEC(ini + i * paso) for i in range(n)]

# --- 6. Eje X de fechas legible ----------------------------------------------
import matplotlib.dates as mdates

def eje_fechas(ax):
    """Ajusta el eje X de fechas al rango real de la serie: anios cuando abarca
    anios, meses cuando abarca meses. Evita que se repita la misma etiqueta
    ('2023 2023 2023 ...') cuando el periodo es corto."""
    loc = mdates.AutoDateLocator(minticks=4, maxticks=9)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    return ax

# --- 7. Trazo por serie: si dos curvas se superponen, el trazo las separa -----
ESTILOS_TG = ['-', '--', ':', '-.', (0, (5, 1, 1, 1))]
TRAZO = {'T1': '-', 'T2': '--', 'T3': ':',
         'V1': '-', 'V2': '--', 'V3': ':'}

# --- 5. Tipografia: tamanos grandes para que la figura se lea en el PDF impreso
TAM_BASE = 13          # texto general dentro de la figura
TAM_EJE  = 14          # etiquetas de los ejes X e Y
TAM_TICK = 12          # numeros de los ejes
TAM_TIT  = 14.5        # titulo del panel
TAM_LEY  = 12          # leyenda

plt.rcParams.update({
    # --- color
    'axes.prop_cycle'   : cycler(color=PALETA_TG),
    'image.cmap'        : 'tg_azul',
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'savefig.facecolor' : 'white',
    # --- tipografia (lo que pidio la revision: ejes y etiquetas legibles)
    'font.family'       : 'sans-serif',
    'font.size'         : TAM_BASE,
    'axes.titlesize'    : TAM_TIT,
    'axes.labelsize'    : TAM_EJE,
    'axes.titleweight'  : 'bold',
    'axes.labelweight'  : 'normal',
    'xtick.labelsize'   : TAM_TICK,
    'ytick.labelsize'   : TAM_TICK,
    'legend.fontsize'   : TAM_LEY,
    'legend.title_fontsize': TAM_LEY,
    'figure.titlesize'  : TAM_TIT + 1.5,
    # --- trazos y ejes discretos
    'lines.linewidth'   : 1.8,
    'lines.markersize'  : 5.5,
    'axes.linewidth'    : 0.9,
    'axes.edgecolor'    : '#8A8A8A',
    'axes.labelcolor'   : C_TEXTO,
    'text.color'        : C_TEXTO,
    'xtick.color'       : '#8A8A8A',
    'ytick.color'       : '#8A8A8A',
    'xtick.labelcolor'  : C_TEXTO,
    'ytick.labelcolor'  : C_TEXTO,
    'xtick.major.width' : 0.9,
    'ytick.major.width' : 0.9,
    'grid.color'        : C_GRIS_CLARO,
    'grid.linewidth'    : 0.7,
    'grid.alpha'        : 0.55,
    'legend.frameon'    : True,
    'legend.framealpha' : 0.92,
    'legend.edgecolor'  : '#C9C9C9',
    # --- exportacion para Overleaf: el texto viaja como texto dentro del PDF
    'figure.dpi'        : 110,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'pdf.fonttype'      : 42,
    'ps.fonttype'       : 42,
    'svg.fonttype'      : 'path',
})

if sns is not None:
    sns.set_palette(PALETA_TG)

print('Estilo de figuras aplicado.')
print(f'  Paleta categorica: {", ".join(PALETA_TG)}')
print(f'  Mapas continuos:   tg_azul (secuencial), tg_div (divergente)')
print(f'  Tamanos: ejes {TAM_EJE} pt, numeros {TAM_TICK} pt, leyenda {TAM_LEY} pt')

# --- Nombres que usa este cuaderno, atados a la paleta unica ---------------
AZUL      = C_AZUL      # estacion / medicion en superficie
NARANJA   = C_AMBAR     # satelite / celda que contiene la estacion
GRIS      = C_GRIS      # celdas vecinas
MAPA_CMAP = 'tg_azul'   # degradacion unica para todos los mapas


## 1. Rutas en Google Drive

In [ ]:
# @@RUTAS_TG@@  (marca unica: no editar esta linea)
# =============================================================================
#  DONDE VIVE EL PROYECTO
#
#  En el Drive hay mas de una copia del proyecto: la del semestre anterior y la
#  actual. Antes cada cuaderno buscaba un archivo de datos por todo el Drive y
#  se quedaba con el PRIMERO que aparecia, que no siempre era el de la carpeta
#  correcta; por eso 'figuras', 'figuras_PDF' y 'figuras_SVG' se crearon en la
#  carpeta vieja.
#
#  Ahora la carpeta del proyecto se decide UNA sola vez y de forma explicita:
#  es la carpeta cuyo nombre esta en NOMBRE_CARPETA y cuya ruta contiene
#  CARPETA_PROYECTO. TODO lo que el cuaderno escriba va ahi.
#
#  Si algun dia mueves el proyecto (por ejemplo a 'Proyecto Aplicado IV'),
#  cambias UNA linea: CARPETA_PROYECTO. Y la cambias en los once cuadernos.
# =============================================================================
import os
import subprocess

CARPETA_PROYECTO = 'Proyecto Aplicado III'      # <-- la carpeta actual del semestre
NOMBRE_CARPETA   = 'Codigos Trabajo de Grado'   # <-- nombre de la carpeta de codigo
RAIZ_DRIVE       = '/content/drive/MyDrive'

# Rutas que nunca se usan como carpeta del proyecto, aunque coincidan de nombre.
EXCLUIR_RUTAS = ['/anterior/', '/.Trash', '/.shortcut-targets-by-id/',
                 '/copia de ', '/copy of ']


def _montar_drive():
    try:
        from google.colab import drive
    except Exception:
        return
    if not os.path.isdir(RAIZ_DRIVE):
        drive.mount('/content/drive')
    else:
        print('Drive ya montado.')


def _find(patron, tipo='f', maxdepth=None, raiz=RAIZ_DRIVE, timeout=300):
    cmd = ['find', raiz]
    if maxdepth is not None:
        cmd += ['-maxdepth', str(maxdepth)]
    cmd += ['-type', tipo, '-name', patron]
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except Exception:
        return []
    return [r for r in res.stdout.strip().split('\n') if r]


def _excluida(ruta):
    r = ruta.lower()
    return any(x.lower() in r for x in EXCLUIR_RUTAS)


def localizar_proyecto():
    """Devuelve la carpeta del proyecto. Falla con un mensaje claro si no la
    encuentra o si encuentra mas de una: nunca elige a ciegas."""
    _montar_drive()

    candidatas = _find(NOMBRE_CARPETA, tipo='d', maxdepth=8)
    if not candidatas:
        candidatas = _find(NOMBRE_CARPETA, tipo='d')
    # Segunda pista: cualquier carpeta que contenga los cuadernos de la cadena.
    for nb in _find('0*_*REVISADO.ipynb', tipo='f', maxdepth=9):
        d = os.path.dirname(nb)
        if d not in candidatas:
            candidatas.append(d)

    validas   = [c for c in candidatas if CARPETA_PROYECTO in c and not _excluida(c)]
    descartes = [c for c in candidatas if c not in validas]

    if len(validas) == 1:
        print(f'Carpeta del proyecto: {validas[0]}')
        if descartes:
            print(f'  ({len(descartes)} carpeta(s) descartada(s) por no estar en '
                  f'"{CARPETA_PROYECTO}")')
        return validas[0]

    print('\n' + '=' * 78)
    print('NO SE PUDO DECIDIR LA CARPETA DEL PROYECTO')
    print('=' * 78)
    print(f'  Se busca:  una carpeta "{NOMBRE_CARPETA}" cuya ruta contenga '
          f'"{CARPETA_PROYECTO}"')
    print(f'  Encontradas: {len(candidatas)}')
    for c in candidatas:
        marca = 'SIRVE     ' if c in validas else 'descartada'
        print(f'    [{marca}] {c}')
    print()
    if not validas:
        print('  Que hacer: comprueba el nombre exacto de la carpeta en Drive y ajusta')
        print('  arriba CARPETA_PROYECTO o NOMBRE_CARPETA. Si moviste el proyecto,')
        print('  recuerda cambiarlo en los once cuadernos.')
    else:
        print('  Hay varias carpetas que sirven. Escribe arriba una CARPETA_PROYECTO')
        print('  mas especifica, por ejemplo la ruta completa desde MyDrive.')
    print('=' * 78 + '\n')
    raise FileNotFoundError(
        f'No hay exactamente una carpeta "{NOMBRE_CARPETA}" bajo "{CARPETA_PROYECTO}".')


# --- La carpeta del proyecto y todo lo que cuelga de ella --------------------
RUTA_BASE     = localizar_proyecto()
RUTA_DATOS    = os.path.join(RUTA_BASE, 'datos')
RUTA_ARRAYS   = os.path.join(RUTA_DATOS, 'arrays')
RUTA_MODELOS  = os.path.join(RUTA_BASE, 'modelos')
RUTA_RES      = os.path.join(RUTA_BASE, 'resultados')
RUTA_FIGS     = os.path.join(RUTA_BASE, 'figuras')       # PNG
RUTA_FIGS_SVG = os.path.join(RUTA_BASE, 'figuras_SVG')   # SVG
RUTA_FIGS_PDF = os.path.join(RUTA_BASE, 'figuras_PDF')   # PDF, el de Overleaf

for _d in (RUTA_DATOS, RUTA_ARRAYS, RUTA_MODELOS, RUTA_RES,
           RUTA_FIGS, RUTA_FIGS_SVG, RUTA_FIGS_PDF):
    os.makedirs(_d, exist_ok=True)


def archivo_de_datos(nombre, obligatorio=True, pista=''):
    """Localiza un archivo de ENTRADA.

    Primero lo busca dentro de la carpeta del proyecto. Solo si no esta lo busca
    en el resto del Drive, y en ese caso avisa en grande: significa que ese
    archivo se quedo en la carpeta vieja y conviene copiarlo.
    Lo que el cuaderno ESCRIBE nunca sale de la carpeta del proyecto.
    """
    for sub in ('datos', os.path.join('datos', 'arrays'), '', 'resultados', 'modelos'):
        cand = os.path.join(RUTA_BASE, sub, nombre) if sub else os.path.join(RUTA_BASE, nombre)
        if os.path.exists(cand):
            return cand

    fuera = _find(nombre, tipo='f')
    if fuera:
        fuera.sort(key=lambda r: os.path.getmtime(r), reverse=True)
        elegido = fuera[0]
        print('\n' + '!' * 78)
        print(f'AVISO: "{nombre}" no esta en la carpeta del {CARPETA_PROYECTO}.')
        print(f'  Se leera de: {elegido}')
        print(f'  Copialo a  : {RUTA_DATOS}')
        print('  Mientras siga en dos sitios, corres el riesgo de leer una version vieja.')
        print('!' * 78 + '\n')
        return elegido

    if obligatorio:
        raise FileNotFoundError(
            f'No se encontro "{nombre}" ni en la carpeta del proyecto ni en el resto '
            f'del Drive.\n  Carpeta del proyecto: {RUTA_BASE}\n  {pista}')
    return None


print(f'  datos      : {RUTA_DATOS}')
print(f'  figuras    : {RUTA_FIGS} (+ _SVG y _PDF)')
print(f'  modelos    : {RUTA_MODELOS}')
print(f'  resultados : {RUTA_RES}')

# --- Entradas propias de este cuaderno ---------------------------------------
# Este cuaderno NO depende del cuaderno 02: usa la medicion real de la estacion.
archivo_de_datos('Datos_2013_2026_completo.csv',
                 pista='Es el CSV de la estacion CESMAG, sin procesar.')


### Figuras en PNG, SVG y PDF

**Añadido en la revisión.** Todas las figuras de este cuaderno se guardan ahora en los tres
formatos, con el mismo nombre y en tres carpetas paralelas: `figuras/` para el PNG,
`figuras_SVG/` para el vectorial editable y `figuras_PDF/` para el que se incluye en Overleaf.

En LaTeX se incluye sin extensión, y así el compilador elige el formato adecuado:

```latex
\includegraphics[width=\linewidth]{figuras_PDF/fig_01b_distribucion}
```

In [ ]:
# --- Guardado de figuras en los tres formatos ----------------------------
#
# El documento se compila en Overleaf, que incluye las figuras en PDF, pero
# conviene tener tambien el SVG (editable) y el PNG (vista rapida y para
# presentaciones). Antes cada cuaderno guardaba en un formato distinto: unos en
# SVG, otros en PDF, otros en PNG. Ahora todos usan esta funcion y producen los
# tres, con el mismo nombre, en tres carpetas paralelas:
#
#   figuras/       nombre.png   300 ppp
#   figuras_SVG/   nombre.svg   vectorial editable
#   figuras_PDF/   nombre.pdf   el que se incluye en LaTeX
#
# En LaTeX basta con \includegraphics{figuras_PDF/nombre} sin extension.
import os
import matplotlib.pyplot as plt

FORMATOS_FIGURA = (
    ('RUTA_FIGS',     'png', {'dpi': 300}),
    ('RUTA_FIGS_SVG', 'svg', {}),
    ('RUTA_FIGS_PDF', 'pdf', {}),
)


def guardar_figura(fig, nombre, verbose=True):
    """Guarda la figura en PNG, SVG y PDF. 'nombre' va SIN extension."""
    if fig is None:
        fig = plt.gcf()
    base = os.path.splitext(os.path.basename(str(nombre)))[0]
    rutas = []
    for var, ext, extra in FORMATOS_FIGURA:
        carpeta = globals().get(var)
        if carpeta is None:
            continue
        os.makedirs(carpeta, exist_ok=True)
        ruta = os.path.join(carpeta, f'{base}.{ext}')
        fig.savefig(ruta, bbox_inches='tight', **extra)
        rutas.append(ruta)
    if verbose:
        print(f'  Figura guardada: {base}.png, .svg y .pdf')
    return rutas


print('Funcion guardar_figura lista: cada figura se guarda en PNG, SVG y PDF.')
for _v in ('RUTA_FIGS', 'RUTA_FIGS_SVG', 'RUTA_FIGS_PDF'):
    print(f'  {_v:<14} {globals().get(_v)}')

## 2. Parámetros de la descarga regional

La interfaz del proyecto NASA POWER admite consultas sobre un rectángulo de coordenadas
además de la consulta por punto. El rectángulo no puede exceder **4.5° × 4.5°**, con un
máximo de 100 celdas por consulta.

Se define una caja de 4° × 4° centrada en la celda que contiene la estación. Sobre una malla
de un grado esto arroja 25 celdas: la de la estación y dos anillos de vecinas.

In [ ]:
# Coordenadas de la estacion CESMAG (las mismas del cuaderno 00)
LAT_EST = 1.2136
LON_EST = -77.2986

# Caja de 4 x 4 grados centrada en la celda de la estacion.
# Sobre la malla de 1 grado los centros caen en los medios grados,
# de modo que la estacion queda en la celda centrada en 1.5 N, 77.5 O.
LAT_MIN, LAT_MAX = -0.5,  3.5
LON_MIN, LON_MAX = -79.5, -75.5

# Solo la variable de irradiancia total: es la que interesa caracterizar
# espacialmente y la que define la resolucion de 1 grado.
VARIABLE = 'ALLSKY_SFC_SW_DWN'

FECHA_INI = '20130801'
FECHA_FIN = '20260430'

URL_REGIONAL_DIARIA = 'https://power.larc.nasa.gov/api/temporal/daily/regional'

print(f'Caja       : lat [{LAT_MIN}, {LAT_MAX}]  lon [{LON_MIN}, {LON_MAX}]')
print(f'Tamanio    : {LAT_MAX-LAT_MIN} x {LON_MAX-LON_MIN} grados  (limite 4.5 x 4.5)')
print(f'Variable   : {VARIABLE}')
print(f'Periodo    : {FECHA_INI} a {FECHA_FIN}')


## 3. Descarga por año

La consulta regional se fracciona por año. Fraccionar tiene dos ventajas: mantiene cada
respuesta en un tamaño manejable y permite reintentar solo el año que falle, sin repetir
toda la descarga.

In [ ]:
def descargar_anio_regional(anio, max_reintentos=3):
    """Descarga un anio de la variable de irradiancia sobre la caja definida.

    Devuelve un DataFrame largo con columnas: lat, lon, fecha, valor.
    Devuelve None si el anio falla tras los reintentos.
    """
    ini = f'{anio}0101'
    fin = f'{anio}1231'
    if anio == int(FECHA_INI[:4]):
        ini = FECHA_INI
    if anio == int(FECHA_FIN[:4]):
        fin = FECHA_FIN

    params = {
        'parameters'   : VARIABLE,
        'community'    : 'RE',
        'latitude-min' : LAT_MIN,
        'latitude-max' : LAT_MAX,
        'longitude-min': LON_MIN,
        'longitude-max': LON_MAX,
        'start'        : ini,
        'end'          : fin,
        'format'       : 'JSON',
    }

    for intento in range(1, max_reintentos + 1):
        try:
            resp = requests.get(URL_REGIONAL_DIARIA, params=params, timeout=300)
            resp.raise_for_status()
            datos = resp.json()

            filas = []
            for feat in datos['features']:
                lon, lat = feat['geometry']['coordinates'][0], feat['geometry']['coordinates'][1]
                serie = feat['properties']['parameter'][VARIABLE]
                for clave, valor in serie.items():
                    filas.append((lat, lon, clave, valor))

            df = pd.DataFrame(filas, columns=['lat', 'lon', 'fecha', 'valor'])
            df['fecha'] = pd.to_datetime(df['fecha'], format='%Y%m%d')
            df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
            df.loc[df['valor'] <= -999, 'valor'] = np.nan

            n_celdas = df[['lat', 'lon']].drop_duplicates().shape[0]
            print(f'  OK {anio}: {len(df):>7,} registros en {n_celdas} celdas')
            return df

        except Exception as e:
            print(f'  Intento {intento}/{max_reintentos} fallido en {anio}: {type(e).__name__}')
            if intento < max_reintentos:
                time.sleep(10)
    print(f'  ERROR: {anio} no se pudo descargar')
    return None


# PARCHE DE LA REVISION: la descarga se reutiliza.
# Bajar catorce anios tarda entre veinte minutos y una hora. Si el archivo ya
# esta en datos/, se lee de ahi. Para volver a bajarlo, pon REDESCARGAR = True.
REDESCARGAR  = False
RUTA_GRILLAS = os.path.join(RUTA_DATOS, 'nasa_power_grillas_diario.csv')

if os.path.exists(RUTA_GRILLAS) and not REDESCARGAR:
    df_grillas = pd.read_csv(RUTA_GRILLAS, parse_dates=['fecha'])
    print('Se reutiliza la descarga guardada (no se vuelve a llamar a NASA POWER).')
    print(f'  Archivo : {RUTA_GRILLAS}')
    print(f'  Registros: {len(df_grillas):,} | '
          f'Celdas: {df_grillas[["lat","lon"]].drop_duplicates().shape[0]} | '
          f'{df_grillas["fecha"].min().date()} a {df_grillas["fecha"].max().date()}')
    print('  Para descargarlo otra vez, pon REDESCARGAR = True arriba.')
else:
    ANIOS = list(range(int(FECHA_INI[:4]), int(FECHA_FIN[:4]) + 1))
    print(f'Descargando {len(ANIOS)} anios...')
    print('-' * 60)

    partes = []
    for anio in ANIOS:
        parte = descargar_anio_regional(anio)
        if parte is not None:
            partes.append(parte)
        time.sleep(3)

    if not partes:
        raise RuntimeError('No se descargo ningun anio. Revisa la conexion y los parametros.')

    df_grillas = pd.concat(partes, ignore_index=True).sort_values(['lat', 'lon', 'fecha'])
    print('-' * 60)
    print(f'Total: {len(df_grillas):,} registros')
    print(f'Celdas: {df_grillas[["lat","lon"]].drop_duplicates().shape[0]}')
    print(f'Periodo: {df_grillas["fecha"].min().date()} a {df_grillas["fecha"].max().date()}')

    df_grillas.to_csv(RUTA_GRILLAS, index=False)
    print('Guardado: nasa_power_grillas_diario.csv  (en kWh/m2 por dia, como llega)')

# --- UNIDADES ---------------------------------------------------------------
# PARCHE DE LA REVISION. El producto DIARIO de NASA POWER entrega
# ALLSKY_SFC_SW_DWN en kWh/m2 por dia, no en W/m2. Se nota en la magnitud: los
# valores llegan entre 3 y 5, que es el rango normal en kWh/m2/dia y seria
# absurdo en W/m2. La version anterior los rotulaba como W/m2 y los comparaba
# con la serie de la estacion, que si esta en W/m2: el sesgo y el RMSE que salian
# de ahi (-138 y 150) no median ningun error fisico, median la diferencia de
# unidades. La correlacion no se veia afectada, porque no depende de la escala.
#
# La serie de la estacion es un promedio diario sobre las 24 horas, asi que la
# conversion correcta a esa misma base es:
#       W/m2 (promedio de 24 h) = kWh/m2/dia * 1000 / 24
if 'valor_kwh_dia' not in df_grillas.columns:
    df_grillas = df_grillas.rename(columns={'valor': 'valor_kwh_dia'})
    df_grillas['valor'] = df_grillas['valor_kwh_dia'] * 1000.0 / 24.0

print()
print('UNIDADES')
print('-' * 60)
print(f'  Como llega de NASA POWER : {df_grillas["valor_kwh_dia"].mean():6.2f} kWh/m2 por dia (media)')
print(f'  Convertido para comparar : {df_grillas["valor"].mean():6.1f} W/m2 (promedio de 24 h)')
print('  A partir de aqui todo el cuaderno trabaja en W/m2.')


## 4. Verificación de la resolución nativa

Este es el paso que sustenta la afirmación del Capítulo 4 sobre la resolución espacial de la
fuente. Se hacen dos comprobaciones:

1. **Espaciado de la malla devuelta.** Se listan las coordenadas únicas y se calcula la
   distancia entre centros consecutivos.
2. **Repetición de valores.** Si la interfaz devuelve la malla meteorológica, más fina que la
   solar, entonces varias celdas contiguas compartirán exactamente el mismo valor de
   irradiancia, porque proceden de la misma celda de un grado del producto de origen. Detectar
   esa repetición es la prueba directa de que la resolución nativa de la irradiancia es de un
   grado, con independencia de la malla en que se entregue.

In [ ]:
lats = np.sort(df_grillas['lat'].unique())
lons = np.sort(df_grillas['lon'].unique())

print('COORDENADAS DEVUELTAS POR LA FUENTE')
print('=' * 60)
print(f'  Latitudes  ({len(lats)}): {lats}')
print(f'  Longitudes ({len(lons)}): {lons}')
paso_lat = np.round(np.diff(lats).mean(), 4) if len(lats) > 1 else np.nan
paso_lon = np.round(np.diff(lons).mean(), 4) if len(lons) > 1 else np.nan
print(f'  Paso en latitud  : {paso_lat} grados')
print(f'  Paso en longitud : {paso_lon} grados')

# Comprobacion de valores repetidos entre celdas vecinas.
piv = df_grillas.pivot_table(index='fecha', columns=['lat', 'lon'], values='valor')
corr_celdas = piv.corr()
n_identicas = 0
pares_identicos = []
cols = list(piv.columns)
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        a, b = piv[cols[i]], piv[cols[j]]
        if np.allclose(a.dropna().values, b.reindex(a.dropna().index).values,
                       rtol=0, atol=1e-6, equal_nan=True):
            n_identicas += 1
            pares_identicos.append((cols[i], cols[j]))

print()
print('REPETICION DE VALORES ENTRE CELDAS')
print('=' * 60)
print(f'  Pares de celdas con serie identica: {n_identicas} de {len(cols)*(len(cols)-1)//2}')
if n_identicas:
    print('  Interpretacion: la irradiancia se origina en una malla mas gruesa que la')
    print('  devuelta. Los primeros pares identicos son:')
    for p in pares_identicos[:5]:
        print('   ', p)
else:
    print('  Interpretacion: cada celda devuelta tiene serie propia, de modo que la malla')
    print('  entregada coincide con la resolucion nativa del producto.')

# Numero de celdas de irradiancia realmente distintas
celdas_unicas = piv.T.round(6).drop_duplicates().shape[0]
print()
print(f'  Celdas con serie de irradiancia distinta: {celdas_unicas} de {len(cols)} devueltas')
# PARCHE DE LA REVISION: antes la resolucion se estimaba repartiendo el area de la
# caja entre el numero de celdas. Eso daba 0.800 grados y contradecia al paso medido
# justo arriba (1.0 grados). El error es que la caja delimita los CENTROS de las
# celdas, no sus bordes: con 5 centros separados 1 grado se cubren 5 grados, no 4.
# El paso entre centros, que ya se midio, ES la resolucion. No hay que estimarla.
print(f'  RESOLUCION NATIVA MEDIDA: {paso_lat:.3g} x {paso_lon:.3g} grados de lado')
print(f'                            = {paso_lat*110.574:.0f} x '
      f'{paso_lon*111.320*np.cos(np.radians(LAT_EST)):.0f} km en la latitud de Pasto')


In [ ]:
# Figura de verificacion: corte en latitud fija mostrando como cambia el valor
# medio con la longitud. Los escalones revelan el tamanio real de la celda.
media_celda = df_grillas.groupby(['lat', 'lon'])['valor'].mean().reset_index()

lat_ref = lats[np.argmin(np.abs(lats - LAT_EST))]
corte   = media_celda[media_celda['lat'] == lat_ref].sort_values('lon')

fig, ax = plt.subplots(figsize=(9, 4))
ax.step(corte['lon'], corte['valor'], where='mid', color=AZUL, lw=2)
ax.plot(corte['lon'], corte['valor'], 'o', color=AZUL, ms=6)
ax.axvline(LON_EST, color=NARANJA, lw=2, ls='--',
           label='Longitud de la estacion CESMAG')
ax.set_xlabel('Longitud (grados)')
ax.set_ylabel('Irradiancia diaria promedio (W/m$^2$)')
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()
guardar_figura(fig, 'fig_08c_verificacion_resolucion')
plt.show()
print('Guardado: fig_08c_verificacion_resolucion.pdf')
print()
print(f'Corte tomado en la latitud {lat_ref} grados.')
print('Cada escalon horizontal corresponde al ancho de una celda del producto de origen.')


## 5. Estadísticas de la serie de irradiancia en cada celda

Para cada celda se calculan las estadísticas que pidió la dirección: promedio, percentil 25,
percentil 75 y variabilidad. La variabilidad se reporta con dos medidas complementarias:

- **Coeficiente de variación**, que es la desviación estándar dividida por la media. Es
  adimensional y permite comparar celdas con niveles medios distintos.
- **Rango intercuartílico**, que es la distancia entre el percentil 75 y el percentil 25. Es
  robusto frente a valores extremos y se expresa en las mismas unidades que la variable.

Todas se calculan sobre la serie **diaria**, que es la escala a la que la irradiancia
satelital tiene sentido comparativo y la misma que emplea la figura de serie temporal del
capítulo de resultados.

In [ ]:
def estadisticas_celda(g):
    v = g['valor'].dropna()
    return pd.Series({
        'n_dias'   : len(v),
        'media'    : v.mean(),
        'mediana'  : v.median(),
        'desv_est' : v.std(),
        'cv_pct'   : v.std() / v.mean() * 100 if v.mean() else np.nan,
        'p25'      : v.quantile(0.25),
        'p75'      : v.quantile(0.75),
        'ric'      : v.quantile(0.75) - v.quantile(0.25),
        'minimo'   : v.min(),
        'maximo'   : v.max(),
    })

stats_celdas = (df_grillas.groupby(['lat', 'lon'])
                          .apply(estadisticas_celda)
                          .reset_index())

# Etiqueta legible de cada celda, en notacion geografica
def etiqueta(lat, lon):
    ns = 'N' if lat >= 0 else 'S'
    eo = 'E' if lon >= 0 else 'O'
    return f'{abs(lat):.1f}{ns}, {abs(lon):.1f}{eo}'

stats_celdas['celda'] = [etiqueta(a, b) for a, b in zip(stats_celdas['lat'], stats_celdas['lon'])]

# Celda que contiene la estacion
lat_celda_est = lats[np.argmin(np.abs(lats - LAT_EST))]
lon_celda_est = lons[np.argmin(np.abs(lons - LON_EST))]
CELDA_ESTACION = (lat_celda_est, lon_celda_est)

print('ESTADISTICAS POR CELDA — irradiancia diaria promedio (W/m2)')
print('=' * 78)
print(stats_celdas[['celda','n_dias','media','p25','p75','ric','desv_est','cv_pct']]
      .round(2).to_string(index=False))
print()
print(f'Celda que contiene la estacion CESMAG: {etiqueta(*CELDA_ESTACION)}')

stats_celdas.to_csv(os.path.join(RUTA_DATOS, 'tabla_08_estadisticas_por_celda.csv'), index=False)
print('Guardado: tabla_08_estadisticas_por_celda.csv')


## 6. Mapa de celdas

Cuatro paneles con la misma malla y la misma degradación de color, cambiando solo la
estadística representada. La estación se marca con un punto y su celda con un borde grueso.

Se intenta superponer el contorno de los departamentos de Colombia. Si la descarga del
contorno falla, la figura se genera igual, solo que sin el contorno; la malla y los valores
son lo esencial.

In [ ]:
# Contorno administrativo. La fuente antigua (un repositorio de GitHub que ya no
# responde) dejaba el mapa sin geografia. Ahora se prueban varias fuentes y, si
# ninguna responde, el mapa sale igual pero avisando.
contorno = None
try:
    import geopandas as gpd
    for _url in ('https://raw.githubusercontent.com/nvkelso/natural-earth-vector/'
                 'master/geojson/ne_50m_admin_1_states_provinces.geojson',
                 'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/'
                 'master/geojson/ne_110m_admin_0_countries.geojson'):
        try:
            _g = gpd.read_file(_url)
            _g = _g.cx[LON_MIN - 2: LON_MAX + 2, LAT_MIN - 2: LAT_MAX + 2]
            if len(_g):
                contorno = _g
                print(f'Contorno cargado: {len(contorno)} poligonos en la zona')
                break
        except Exception:
            continue
except Exception:
    pass
if contorno is None:
    print('No se pudo cargar ningun contorno administrativo.')
    print('El mapa se dibuja sin fronteras. No afecta a los valores representados.')


PANELES = [
    ('media',  'Irradiancia diaria promedio (W/m$^2$)'),
    ('p25',    'Percentil 25 de la irradiancia diaria (W/m$^2$)'),
    ('p75',    'Percentil 75 de la irradiancia diaria (W/m$^2$)'),
    ('cv_pct', 'Coeficiente de variacion (%)'),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 12))
axes = axes.ravel()

for ax, (col, etiq) in zip(axes, PANELES):
    vmin, vmax = stats_celdas[col].min(), stats_celdas[col].max()
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap(MAPA_CMAP)

    if contorno is not None:
        contorno.boundary.plot(ax=ax, color=C_GRIS, lw=0.6, zorder=1)

    for _, fila in stats_celdas.iterrows():
        x0 = fila['lon'] - paso_lon / 2
        y0 = fila['lat'] - paso_lat / 2
        es_estacion = (fila['lat'] == CELDA_ESTACION[0]) and (fila['lon'] == CELDA_ESTACION[1])
        ax.add_patch(Rectangle(
            (x0, y0), paso_lon, paso_lat,
            facecolor=cmap(norm(fila[col])),
            edgecolor='black' if es_estacion else 'white',
            lw=2.4 if es_estacion else 0.7,
            alpha=0.88, zorder=2
        ))
        ax.text(fila['lon'], fila['lat'], f"{fila[col]:.0f}",
                ha='center', va='center', fontsize=11,
                color='black', zorder=4)

    ax.plot(LON_EST, LAT_EST, marker='*', ms=17, color=AZUL,
            markeredgecolor='white', markeredgewidth=1.0, zorder=5,
            label='Estacion CESMAG')

    ax.set_xlim(LON_MIN - paso_lon, LON_MAX + paso_lon)
    ax.set_ylim(LAT_MIN - paso_lat, LAT_MAX + paso_lat)
    ax.set_xlabel('Longitud (grados)')
    ax.set_ylabel('Latitud (grados)')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.25, ls=':')
    ax.legend(loc='upper right', frameon=True, framealpha=0.9)

    sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label(etiq, fontsize=12)

fig.tight_layout()
guardar_figura(fig, 'fig_08a_mapa_estadisticas')
plt.show()
print('Guardado: fig_08a_mapa_estadisticas.pdf')


## Mapas de la zona con la malla de NASA POWER

Los dos mapas que siguen son los que pidio la direccion: la zona de estudio de verdad —con relieve, fronteras y ciudades— y la malla del producto satelital dibujada encima. El segundo es un acercamiento a la celda que contiene la estacion y es el que sostiene la correccion del documento sobre la resolucion del servicio.


In [ ]:
# =============================================================================
#  MAPAS DE LA ZONA CON LA MALLA DE NASA POWER
#
#  Peticion de la direccion: mapas de verdad de la zona, con las grillas encima.
#  La version anterior dibujaba solo cuadros de colores sobre unos ejes de
#  latitud y longitud, sin geografia: la descarga del contorno fallaba y el mapa
#  quedaba sin costa, sin fronteras y sin ciudades.
#
#  Este bloque construye el mapa por capas, y cada capa es opcional: si una no
#  se puede descargar, el mapa sigue saliendo con las demas y avisa cual falto.
#    1. Mapa base de relieve (teselas)      -> contexto visual
#    2. Fronteras departamentales/estatales -> referencia administrativa
#    3. Malla de NASA POWER                 -> lo que se quiere mostrar
#    4. Estacion y ciudades de referencia   -> para ubicarse
#    5. Barra de escala en kilometros       -> para dimensionar la celda
# =============================================================================
import math
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# --- Ciudades de referencia (centros urbanos aproximados) --------------------
# Pasto no entra: la estrella de la estacion ya lo marca.
CIUDADES = [
    ('Ipiales',  0.8256, -77.6444),
    ('Tumaco',   1.7986, -78.8156),
    ('Mocoa',    1.1519, -76.6483),
    ('Popayan',  2.4448, -76.6147),
    ('Cali',     3.4516, -76.5320),
]

_AVISOS = []

def cargar_contorno():
    """Fronteras administrativas. Prueba varias fuentes y devuelve la primera
    que responda; si ninguna responde, devuelve None y el mapa sale sin ellas."""
    try:
        import geopandas as gpd
    except Exception:
        _AVISOS.append('geopandas no esta instalado: sin fronteras.')
        return None
    FUENTES = [
        ('Natural Earth 50m (departamentos y estados)',
         'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/'
         'master/geojson/ne_50m_admin_1_states_provinces.geojson'),
        ('Natural Earth 110m (paises)',
         'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/'
         'master/geojson/ne_110m_admin_0_countries.geojson'),
    ]
    for nombre, url in FUENTES:
        try:
            g = gpd.read_file(url)
            g = g.cx[LON_MIN - 2: LON_MAX + 2, LAT_MIN - 2: LAT_MAX + 2]
            if len(g):
                print(f'Fronteras cargadas: {nombre}  ({len(g)} poligonos en la zona)')
                return g
        except Exception as e:
            _AVISOS.append(f'{nombre}: {type(e).__name__}')
    _AVISOS.append('Ninguna fuente de fronteras respondio.')
    return None


def poner_mapa_base(ax):
    """Relieve de fondo. Es lo que hace que la figura se lea como un mapa y no
    como una cuadricula. Si no hay teselas, no pasa nada: el mapa sigue."""
    try:
        import contextily as cx
    except Exception:
        _AVISOS.append('contextily no esta instalado: sin relieve de fondo.')
        return False
    PROVEEDORES = [
        ('relieve Esri',      lambda: cx.providers.Esri.WorldShadedRelief),
        ('terreno Esri',      lambda: cx.providers.Esri.WorldTerrain),
        ('OpenStreetMap',     lambda: cx.providers.OpenStreetMap.Mapnik),
    ]
    for nombre, prov in PROVEEDORES:
        try:
            cx.add_basemap(ax, crs='EPSG:4326', source=prov(), attribution_size=7)
            return True
        except Exception as e:
            _AVISOS.append(f'mapa base {nombre}: {type(e).__name__}')
    return False


def barra_escala(ax, lat_ref, largo_km=100, x=0.06, y=0.06):
    """Barra de escala en kilometros, dibujada en coordenadas del eje."""
    km_por_grado_lon = 111.320 * math.cos(math.radians(lat_ref))
    largo_grados = largo_km / km_por_grado_lon
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    px = x0 + x * (x1 - x0); py = y0 + y * (y1 - y0)
    alto = 0.012 * (y1 - y0)
    ax.add_patch(Rectangle((px, py), largo_grados/2, alto, facecolor=C_TEXTO,
                           edgecolor=C_TEXTO, lw=0.8, zorder=8))
    ax.add_patch(Rectangle((px + largo_grados/2, py), largo_grados/2, alto,
                           facecolor='white', edgecolor=C_TEXTO, lw=0.8, zorder=8))
    ax.text(px + largo_grados / 2, py + 0.035 * (y1 - y0), f'{largo_km} km',
            ha='center', va='bottom', fontsize=11, color=C_TEXTO, zorder=9,
            bbox=dict(boxstyle='round,pad=0.18', fc='white', ec='none', alpha=0.75))


def marcar_ciudades(ax, solo_dentro=True):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    for nombre, la, lo in CIUDADES:
        if solo_dentro and not (x0 <= lo <= x1 and y0 <= la <= y1):
            continue
        ax.plot(lo, la, marker='o', ms=6, color=C_TEXTO, mec='white', mew=1.0, zorder=7)
        ax.text(lo + 0.06, la + 0.06, nombre, fontsize=11, color=C_TEXTO,
                zorder=8, bbox=dict(boxstyle='round,pad=0.16', fc='white',
                                    ec='none', alpha=0.72))


CONTORNO = cargar_contorno()
KM_LAT = 110.574
KM_LON = 111.320 * math.cos(math.radians(LAT_EST))
print(f'Tamanio de una celda: {paso_lat:.3g}o x {paso_lon:.3g}o  '
      f'= {paso_lat*KM_LAT:.0f} km x {paso_lon*KM_LON:.0f} km')


In [ ]:
# --- MAPA 1: la zona con la malla de NASA POWER encima -----------------------
fig, ax = plt.subplots(figsize=(11, 10.5))

ax.set_xlim(LON_MIN - paso_lon * 0.75, LON_MAX + paso_lon * 0.75)
ax.set_ylim(LAT_MIN - paso_lat * 0.75, LAT_MAX + paso_lat * 0.75)
ax.set_aspect(1 / math.cos(math.radians(LAT_EST)))   # proporcion real en la latitud de Pasto

hay_base = poner_mapa_base(ax)
if CONTORNO is not None:
    CONTORNO.boundary.plot(ax=ax, color=C_GRIS, lw=0.9, zorder=2)

# La malla: una celda por cada punto que devolvio NASA POWER
for _, fila in stats_celdas.iterrows():
    es_est = (fila['lat'] == CELDA_ESTACION[0]) and (fila['lon'] == CELDA_ESTACION[1])
    ax.add_patch(Rectangle(
        (fila['lon'] - paso_lon / 2, fila['lat'] - paso_lat / 2), paso_lon, paso_lat,
        facecolor=C_AZUL if es_est else 'none',
        alpha=0.20 if es_est else 1.0,
        edgecolor=C_AZUL_OSCURO if es_est else C_AZUL,
        lw=2.8 if es_est else 1.1, zorder=3))
    ax.plot(fila['lon'], fila['lat'], marker='+', ms=7, color=C_AZUL,
            mew=1.2, zorder=4)

ax.plot(LON_EST, LAT_EST, marker='*', ms=20, color=C_AMBAR,
        markeredgecolor='white', markeredgewidth=1.4, zorder=8)
ax.annotate('Estacion CESMAG (Pasto)', xy=(LON_EST, LAT_EST),
            xytext=(18, 22), textcoords='offset points',
            fontsize=12, fontweight='bold', color=C_TEXTO, zorder=9,
            bbox=dict(boxstyle='round,pad=0.28', fc='white', ec=C_AMBAR, lw=1.4, alpha=0.92),
            arrowprops=dict(arrowstyle='-', color=C_AMBAR, lw=1.6))

marcar_ciudades(ax)
barra_escala(ax, LAT_EST, largo_km=100)

ax.set_xlabel('Longitud (grados)')
ax.set_ylabel('Latitud (grados)')
leyenda = [
    Line2D([], [], color=C_AZUL, lw=1.4, label=f'Celda de NASA POWER '
           f'({paso_lat*KM_LAT:.0f} x {paso_lon*KM_LON:.0f} km)'),
    Line2D([], [], color=C_AZUL_OSCURO, lw=2.8, label='Celda que contiene la estacion'),
    Line2D([], [], color=C_AMBAR, marker='*', ms=15, ls='none',
           markeredgecolor='white', label='Estacion CESMAG'),
    Line2D([], [], color=C_TEXTO, marker='o', ms=6, ls='none',
           markeredgecolor='white', label='Ciudades de referencia'),
]
ax.legend(handles=leyenda, loc='upper center', bbox_to_anchor=(0.5, -0.09),
          ncol=2, fontsize=11, frameon=False)
plt.tight_layout()
guardar_figura(fig, 'fig_08d_mapa_zona')
plt.show()

if _AVISOS:
    print('Capas que no se pudieron descargar (el mapa sale igual, sin ellas):')
    for a in dict.fromkeys(_AVISOS):
        print('   -', a)
print('Guardado: fig_08d_mapa_zona (png, svg y pdf)')


In [ ]:
# --- MAPA 2: acercamiento a la celda de la estacion --------------------------
# Este es el mapa que sostiene la correccion del documento. La tesis decia que
# NASA POWER entrega valores "interpolados al punto de coordenadas consultado".
# No es asi: devuelve el valor de la celda que contiene esas coordenadas. Aqui se
# ve: varios puntos separados decenas de kilometros caen dentro de la MISMA
# celda y reciben, por tanto, exactamente el mismo dato.
lat_e, lon_e = CELDA_ESTACION
margen = paso_lat * 0.85

fig, ax = plt.subplots(figsize=(10.5, 9.5))
ax.set_xlim(lon_e - paso_lon / 2 - margen, lon_e + paso_lon / 2 + margen)
ax.set_ylim(lat_e - paso_lat / 2 - margen, lat_e + paso_lat / 2 + margen)
ax.set_aspect(1 / math.cos(math.radians(LAT_EST)))

poner_mapa_base(ax)
if CONTORNO is not None:
    CONTORNO.boundary.plot(ax=ax, color=C_GRIS, lw=1.0, zorder=2)

for dla in (-1, 0, 1):
    for dlo in (-1, 0, 1):
        centro_la, centro_lo = lat_e + dla * paso_lat, lon_e + dlo * paso_lon
        es_est = (dla == 0 and dlo == 0)
        ax.add_patch(Rectangle(
            (centro_lo - paso_lon / 2, centro_la - paso_lat / 2), paso_lon, paso_lat,
            facecolor=C_AZUL if es_est else 'none', alpha=0.16 if es_est else 1.0,
            edgecolor=C_AZUL_OSCURO if es_est else C_AZUL,
            lw=3.0 if es_est else 1.2, zorder=3))

# Puntos de prueba dentro de la celda de la estacion: todos devuelven lo mismo
pruebas = [
    ('Estacion CESMAG',      LAT_EST,               LON_EST),
    ('Esquina noroeste',     lat_e + paso_lat*0.45, lon_e - paso_lon*0.45),
    ('Esquina sureste',      lat_e - paso_lat*0.45, lon_e + paso_lon*0.45),
    ('Centro de la celda',   lat_e,                 lon_e),
]
for nombre, la, lo in pruebas:
    ax.plot(lo, la, marker='o', ms=9, color=C_AMBAR, mec='white', mew=1.3, zorder=7)
d_lat = (pruebas[1][1] - pruebas[2][1]) * KM_LAT
d_lon = (pruebas[1][2] - pruebas[2][2]) * KM_LON
sep_km = math.hypot(d_lat, d_lon)
ax.annotate('', xy=(pruebas[1][2], pruebas[1][1]), xytext=(pruebas[2][2], pruebas[2][1]),
            arrowprops=dict(arrowstyle='<->', color=C_TEJA, lw=2.0), zorder=6)
ax.text(lon_e - paso_lon*0.27, lat_e - paso_lat*0.22,
        f'{sep_km:.0f} km de separacion,\nel MISMO valor de irradiancia',
        ha='center', va='top', fontsize=12.5, fontweight='bold', color=C_TEJA, zorder=9,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=C_TEJA, lw=1.4, alpha=0.93))

ax.plot(LON_EST, LAT_EST, marker='*', ms=20, color=C_AMBAR,
        markeredgecolor='white', markeredgewidth=1.4, zorder=8)
marcar_ciudades(ax)
barra_escala(ax, LAT_EST, largo_km=25)
ax.set_xlabel('Longitud (grados)')
ax.set_ylabel('Latitud (grados)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.09), ncol=2,
          frameon=False, fontsize=11, handles=[
    Line2D([], [], color=C_AZUL_OSCURO, lw=3.0,
           label=f'Celda que contiene la estacion ({paso_lat*KM_LAT:.0f} x {paso_lon*KM_LON:.0f} km)'),
    Line2D([], [], color=C_AZUL, lw=1.2, label='Celdas vecinas'),
    Line2D([], [], color=C_AMBAR, marker='o', ms=9, ls='none', mec='white',
           label='Puntos que devuelven el mismo dato'),
])
plt.tight_layout()
guardar_figura(fig, 'fig_08e_zoom_celda_estacion')
plt.show()
print('Guardado: fig_08e_zoom_celda_estacion (png, svg y pdf)')
print()
print('LECTURA: cualquier coordenada dentro del rectangulo grueso recibe el mismo')
print(f'valor. La celda mide {paso_lat*KM_LAT:.0f} x {paso_lon*KM_LON:.0f} km, de modo que')
print('"interpolado al punto consultado" no describe lo que entrega el servicio.')


## 7. Series por celda frente a la serie de la estación

Se seleccionan cinco celdas: la que contiene la estación y sus cuatro vecinas inmediatas al
norte, al sur, al este y al oeste. Para cada una se dibuja el promedio diario y su media móvil
de 90 días, en el mismo formato de la figura de serie temporal del capítulo de resultados, y
se superpone la serie de la estación.

La media móvil de 90 días se usa porque suaviza el ruido diario y deja ver la componente
estacional, que es lo que interesa comparar entre celdas.

In [ ]:
# Serie diaria de la estacion, a partir del CSV ORIGINAL de la estacion
#
# PARCHE DE LA REVISION. Antes esta celda leia 'dataset_final_interpolacion.csv',
# que produce el cuaderno 02. Eso tenia dos problemas:
#   1) obligaba a ejecutar el 02 antes que este cuaderno, cuando el plan lo situa
#      en la Fase 1, justo despues del 01;
#   2) y era metodologicamente peor: comparar el satelite contra una serie
#      IMPUTADA es comparar el satelite contra valores que en parte inventamos.
# Ahora se usa la medicion real, con la unica limpieza fisica que no depende de
# ninguna tecnica de imputacion (la misma del cuaderno 02):
#     RS < 0            -> 0
#     RS > 0 de noche   -> NaN   (el sensor no puede medir luz sin sol)
#     noche             -> 0
# Los huecos diurnos se quedan como NaN: un dia con poca cobertura no entra en
# la comparacion, en lugar de entrar con un promedio inventado.
COBERTURA_MINIMA = 0.80        # fraccion del dia con dato valido exigida

_ruta_est = archivo_de_datos('Datos_2013_2026_completo.csv',
                             pista='Es el CSV de la estacion CESMAG, sin procesar.')

with open(_ruta_est, 'r', encoding='utf-8', errors='replace') as _f:
    _lineas = [_f.readline().strip() for _ in range(3)]
_SEP = ';' if _lineas[0].count(';') > _lineas[0].count(',') else ','
_DEC = ',' if (_SEP == ';' and ',' in _lineas[1]) else '.'

df_est = pd.read_csv(_ruta_est, sep=_SEP, decimal=_DEC,
                     parse_dates=['DateTime'], index_col='DateTime')
df_est.columns = ['RS']
df_est.index.name = 'timestamp'
df_est.sort_index(inplace=True)
df_est['RS'] = pd.to_numeric(df_est['RS'], errors='coerce')

_h = df_est.index.hour
_noche = ~((_h >= 6) & (_h <= 18))
df_est.loc[df_est['RS'] < 0, 'RS'] = 0.0                       # negativos -> 0
df_est.loc[(df_est['RS'] > 0) & _noche, 'RS'] = np.nan         # luz de noche -> NaN
df_est.loc[_noche, 'RS'] = df_est.loc[_noche, 'RS'].fillna(0.0)  # noche -> 0

_por_dia   = df_est['RS'].resample('D')
_validos   = _por_dia.count()
_esperados = _por_dia.size()
serie_est_diaria = _por_dia.mean()
serie_est_diaria[_validos / _esperados < COBERTURA_MINIMA] = np.nan

_n_ok = int(serie_est_diaria.notna().sum())
print(f'Serie de la estacion (medicion real, sin imputar):')
print(f'  Periodo    : {serie_est_diaria.index.min().date()} a '
      f'{serie_est_diaria.index.max().date()}')
print(f'  Dias totales                    : {len(serie_est_diaria):,}')
print(f'  Dias con cobertura >= {COBERTURA_MINIMA:.0%}      : {_n_ok:,}')
print(f'  Dias descartados por cobertura  : {len(serie_est_diaria) - _n_ok:,}')

# Seleccion de celdas: la de la estacion y sus cuatro vecinas
lat_e, lon_e = CELDA_ESTACION
CELDAS_SEL = [
    (lat_e,            lon_e,            'Celda de la estacion'),
    (lat_e + paso_lat, lon_e,            'Vecina al norte'),
    (lat_e - paso_lat, lon_e,            'Vecina al sur'),
    (lat_e,            lon_e + paso_lon, 'Vecina al este'),
    (lat_e,            lon_e - paso_lon, 'Vecina al oeste'),
]
CELDAS_SEL = [(a, b, n) for a, b, n in CELDAS_SEL
              if ((df_grillas['lat'] == a) & (df_grillas['lon'] == b)).any()]

print(f'\nCeldas seleccionadas: {len(CELDAS_SEL)}')
for a, b, n in CELDAS_SEL:
    print(f'  {n:<24} {etiqueta(a, b)}')


In [ ]:
fig, axes = plt.subplots(len(CELDAS_SEL), 1, figsize=(13, 3.0 * len(CELDAS_SEL)),
                         sharex=True, sharey=True)
if len(CELDAS_SEL) == 1:
    axes = [axes]

est_ma = serie_est_diaria.rolling(90, center=True, min_periods=30).mean()

for ax, (a, b, nombre) in zip(axes, CELDAS_SEL):
    sub = (df_grillas[(df_grillas['lat'] == a) & (df_grillas['lon'] == b)]
           .set_index('fecha')['valor'].sort_index())
    sub_ma = sub.rolling(90, center=True, min_periods=30).mean()

    es_celda_est = (a == lat_e) and (b == lon_e)
    color_celda  = NARANJA if es_celda_est else GRIS

    ax.plot(sub.index, sub.values, color=color_celda, lw=0.4, alpha=0.35)
    ax.plot(sub_ma.index, sub_ma.values, color=color_celda, lw=2.2,
            label=f'Celda {etiqueta(a, b)}, media movil de 90 dias')
    ax.plot(est_ma.index, est_ma.values, color=AZUL, lw=2.2, ls='--',
            label='Estacion CESMAG, media movil de 90 dias')

    ax.set_ylabel('Irradiancia (W/m$^2$)')
    ax.legend(loc='lower right', ncol=2, frameon=False)
    ax.grid(True, alpha=0.3)
    ax.text(0.008, 0.90, nombre, transform=ax.transAxes,
            fontsize=12, va='top', color=C_TEXTO)

axes[-1].set_xlabel('Fecha')
eje_fechas(axes[-1])
fig.tight_layout()
guardar_figura(fig, 'fig_08b_series_por_celda')
plt.show()
print('Guardado: fig_08b_series_por_celda.pdf')


## 8. Comparación cuantitativa de cada celda con la estación

El mapa y las series dicen cómo se ven las celdas. Esta tabla dice cuál se parece más a la
estación, con tres medidas sobre la serie diaria común a ambas:

- **Correlación de Pearson**, que mide si suben y bajan juntas.
- **Sesgo**, que es el promedio de la diferencia entre la celda y la estación. Positivo
  significa que la celda reporta más irradiancia que el piranómetro.
- **Raíz del error cuadrático medio**, que resume la magnitud típica de la discrepancia.

Nota sobre la correlación: al tratarse de dos series temporales autocorrelacionadas, el
coeficiente se reporta aquí como **medida descriptiva de asociación** y no se acompaña de
prueba de significancia, porque el supuesto de observaciones independientes que esa prueba
requiere no se cumple.

In [ ]:
filas = []
for (a, b), g in df_grillas.groupby(['lat', 'lon']):
    s = g.set_index('fecha')['valor'].sort_index()
    comun = s.index.intersection(serie_est_diaria.index)
    x, y = s.loc[comun], serie_est_diaria.loc[comun]
    val = pd.concat([x, y], axis=1).dropna()
    if len(val) < 100:
        continue
    xc, yc = val.iloc[:, 0], val.iloc[:, 1]
    filas.append({
        'celda'    : etiqueta(a, b),
        'lat'      : a,
        'lon'      : b,
        'n_dias'   : len(val),
        'corr'     : xc.corr(yc),
        'sesgo'    : (xc - yc).mean(),
        'rmse'     : np.sqrt(((xc - yc) ** 2).mean()),
        'media_celda'    : xc.mean(),
        'media_estacion' : yc.mean(),
    })

comp = pd.DataFrame(filas).sort_values('corr', ascending=False)
comp['es_celda_estacion'] = ((comp['lat'] == lat_e) & (comp['lon'] == lon_e))

print('COMPARACION DE CADA CELDA CON LA ESTACION — serie diaria')
print('=' * 88)
print(comp[['celda','n_dias','corr','sesgo','rmse','media_celda','media_estacion',
            'es_celda_estacion']].round(3).to_string(index=False))

comp.to_csv(os.path.join(RUTA_DATOS, 'tabla_08_comparacion_celda_estacion.csv'), index=False)
print()
print('Guardado: tabla_08_comparacion_celda_estacion.csv')

mejor = comp.iloc[0]
celda_est_fila = comp[comp['es_celda_estacion']].iloc[0] if comp['es_celda_estacion'].any() else None

print()
print('LECTURA DEL RESULTADO')
print('-' * 60)
print(f'  Celda con mayor correlacion : {mejor["celda"]}  (r = {mejor["corr"]:.3f})')
if celda_est_fila is not None:
    print(f'  Celda que contiene la estacion: {celda_est_fila["celda"]}  '
          f'(r = {celda_est_fila["corr"]:.3f})')
    if mejor['celda'] == celda_est_fila['celda']:
        print('  La celda que contiene la estacion es tambien la que mejor la reproduce.')
    else:
        print('  La celda que mejor reproduce la estacion NO es la que la contiene.')
        print('  Este hallazgo merece comentario en la discusion: indica que el promedio')
        print('  de area de la celda vecina se aproxima mejor al regimen del emplazamiento.')


## 9. Tabla lista para LaTeX

Genera el cuerpo de la tabla de estadísticas por celda con el formato de cifras declarado en
la introducción del documento: dos decimales para magnitudes en W/m² y porcentajes, coma para
los miles.

In [ ]:
def fmt(x, dec=2):
    """Formatea con el criterio del documento: coma de miles, punto decimal,
    y la coma protegida con llaves para que LaTeX no la trate como signo."""
    s = f'{x:,.{dec}f}'
    return s.replace(',', '{,}')

orden = stats_celdas.sort_values(['lat', 'lon'], ascending=[False, True])

print('% --- Cuerpo de la tabla de estadisticas por celda ---')
for _, f in orden.iterrows():
    marca = r'\textbf{' + f['celda'] + '}' if (f['lat'] == lat_e and f['lon'] == lon_e) else f['celda']
    print(f"{marca} & {fmt(f['media'])} & {fmt(f['p25'])} & {fmt(f['p75'])} & "
          f"{fmt(f['ric'])} & {fmt(f['desv_est'])} & {fmt(f['cv_pct'])} \\\\")

print()
print('% --- Cuerpo de la tabla de comparacion con la estacion ---')
for _, f in comp.iterrows():
    print(f"{f['celda']} & {f['n_dias']:,} & {f['corr']:.3f} & {fmt(f['sesgo'])} & "
          f"{fmt(f['rmse'])} \\\\".replace(f"{f['n_dias']:,}", fmt(f['n_dias'], 0)))


## 10. Resumen

Al terminar este cuaderno tienes:

- La **verificación empírica** de la resolución espacial de la fuente, que es lo que respalda
  la afirmación del Capítulo 4 y la justificación de por qué no había opción de 2.5° o 5°.
- El **mapa de celdas** con promedio, percentil 25, percentil 75 y variabilidad, que responde
  al octavo punto de la reunión.
- Las **series por celda** frente a la de la estación, que responden al noveno.
- La **tabla de comparación**, que permite responder en la sustentación a la pregunta de en
  qué celda cae la estación y cuánto se parece esa celda a lo que el piranómetro mide.

El texto de metodología y de resultados que acompaña a estas figuras está en los archivos
`B3_metodologia_etapa3_espacial.tex` y `B4_resultados_espacial.tex`.